# Entryscape Demo SCDoD

### Purpose 

The notebook demonstrates how to work with open data from the Entryscape platform using Python.

It covers:
- Parsing and inspecting RDF metadata for datasets, and distributions.
- Extracting specific metadata fields (e.g., title, publisher) using namespace-prefixed queries.
- Finding distributions with a specific format (e.g., application/json).
- Discovering API endpoints for data access.
- Querying RowStore APIs with regex-based filters and handling pagination.
- (Visualizing the extracted data)

### Working with the Notebook

To work with this notebook in Jupyter, run each cell step by step using `Shift+Enter` or the (__play__-icon in the menu bar).
This executes the selected cell and moves to the next one.
You can edit code cells, and re-run them as needed.
Follow the notebook from top to bottom for a guided workflow.

Some cells contain commented-out lines only (starting with `#`). For example:
```py
# get_resource_metadata_field(...)
```
These commented-out lines are templates or hints for how to use utility functions to extract specific metadata. Uncomment (remove the `#`) and fill them in with the appropriate arguments to use them in your workflow.

### Utility Functions

This notebook uses custom utility functions from `utils.py`.
You can find explanations for each function and their arguments directly in the file.

### Documentation

[EntryStore Documentation](https://entrystore.org)

[EntryStore API Documentation](https://entrystore.org/api)


In [ ]:
import plotly

import pandas as pd
import plotly.express as px

from rdflib import Graph
from utils import find_distributions, find_api_endpoints, get_rowstore_data, get_metadata_field, get_license_label
from IPython.display import HTML

# Exploring Metadata using RDFLib
Define IDs and catalog URL.

In [ ]:
context_id = "1"
dataset_id = "5"

catalog_url = f"https://zg-demo.entryscape.net/store/{context_id}"

Using rdflib, parse the rdf-graph of a dataset and print it:

In [ ]:
g = Graph()
# Parse dataset RDF
dataset_url = f"{catalog_url}/resource/{dataset_id}"
g.parse(dataset_url)

g.print(format="xml")

# Extracting Metadata from Graph

You can use the ``get_metadata_field`` utility function to extract specific metadata fields from RDF resources.
Provide the metadata-field name (as a QName like "dcterms:title" or a full URI), and either a ``catalog_url`` with ``resource_id``, or a full ``resource_uri``.
Optionally, specify the language (default is "de").

The function returns the value for the specified field and language, or a list if ``all_values=True``.

In [ ]:
# Get the German title of a dataset using catalog_url and resource_id
title = get_metadata_field("dcterms:title", catalog_url, dataset_id, language="de")

# Or, using a full resource URI and English language
title_en = get_metadata_field("dcterms:title", resource_uri="https://zg-demo.entryscape.net/store/1/resource/5", language="en")

print(f"Dataset title (de): {title}")
print(f"Dataset title (en): {title_en}")

Extract other metadata, for example the description the modification date.


Have a look at the metadata-graph printed above or https://zg-demo.entryscape.net/store/1/resource/5.

In [ ]:
# get_resource_metadata_field(...)

Get the name of the Publisher.

1. get the `publisher` URI from the dataset
2. get the `name` of the publisher (note: the publisher entry is a separate resource)

In [ ]:
# publisher_uri = get_metadata_field(...)

# instead of the dataset_id, we use resource_uri=publisher_uri to get the publisher name
# publisher_name = get_metadata_field(...)

Example: Find distributions with format `application/json`

In [ ]:
find_distributions(
    catalog_url,
    dataset_id,
    format_mime="application/json"
)

Get the `license` from distribution metadata.
1. look for the distribution in the dataset's metadata (you can click on one of the links above)
2. get the ``license`` metadata value using ``get_metadata_field``
3. the utility function `get_license_label` fetches the license label from the defined vocabulary https://dcat-ap.ch/vocabulary/licenses'. You can select one of the available languages in the defined vocabulary.

In [ ]:
# licennse_url = get_metadata_field(
#     "...",
#     resource_uri=...
# )

# license_label = get_license_label(licennse_url, language="de")

# Get data from RowStore API


Find api endpoints

In [ ]:
api_endpoints = find_api_endpoints(
    catalog_url,
    dataset_id
)
api_endpoints

You can check out the RowStore API in your browser following one of the links under `page`, e.g., https://swagger.entryscape.com/?url=https://zg-demo.entryscape.net/rowstore/dataset/be3f5870-fee6-40ef-ab38-5d90ba71938f/swagger'

Query parameter (regex) to extract one data point per day at noon

In [ ]:
query_params = {
    "datum/zeit": "^(2022|2023|2024).*12:00:00$" 
    }

Fetch the data from each of the API endpoints using the utility function `get_rowstore_data` and combine fetched data in one data frame

In [ ]:
df_combined = pd.DataFrame()

for api_endpoint in api_endpoints:
    accessURL = api_endpoint.get("accessURL")
    title = api_endpoint.get("title")
    print(f"Fetching data from API for station: {title}")
    gw_data = get_rowstore_data(
        accessURL,
        query_params,
        fetch_all=True
    )

    df = pd.DataFrame(gw_data)
    df['gw-stand [m ü.m.]'] = pd.to_numeric(df['gw-stand [m ü.m.]'], errors='coerce')
    df = df.assign(station=title) # add station column
    df_combined = pd.concat([df_combined, df], ignore_index=True)

### Visualize the data using plotly

Example: Show the difference to the mean gw-stand for each of the stations.

Add a column with the difference to the mean gw-stand per station.

In [ ]:
df_combined['gw-stand-diff'] = df_combined['gw-stand [m ü.m.]'] - df_combined.groupby('station')['gw-stand [m ü.m.]'].transform('mean')
df_combined

In [ ]:
# template = "plotly_dark"
template = "seaborn"

fig = px.line(
    df_combined,
    x="datum/zeit",
    y="gw-stand-diff",
    color="station",
    labels={"datum/zeit": "Datum", "gw-stand-diff": "Grundwasserstand Differenz zum Mittelwert (m)"},
    template=template,
    custom_data=["station"]
    )

# legend title
fig.update_layout(legend_title_text='Messstation')
# addd caption with data source
fig.add_annotation(
    text=(f"Datenquelle: {publisher_name}" if 'publisher_name' in locals() else "Datenquelle: Opendata Zug"),
    xref="paper", yref="paper",
    x=0, y=-0.2,
    showarrow=False,
    font=dict(size=12, color="grey")
)
# change the hover template
fig.update_traces(
    hovertemplate=
    'Datum: %{x|%d.%m.%Y}<br>' +
    'Grundwasserstand (Diff.): %{y:.2f} m<br>' +
    'Messstation: %{customdata[0]} <br>' +
    '<extra></extra>'
)


In [ ]:
# offline plot for renku
HTML(plotly.offline.plot(fig))